In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from diagnostic_reasoning import (
    OVRProjectionResult, ovr_project_posterior, project_ovr_to_coherent,
    _project_ovr_to_coherent, common_reference_posterior, normalize_priors,
    validate_posterior, float_vector, representable_exp, require_candidate,
    CERTAINTY_ESTIMATES, qualitative_score,
)

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, widgets, VBox, HBox, Layout, Output, Label

import scienceplots
plt.style.use('science')
plt.rcParams['text.usetex'] = False


def naive_multinomial_predict_cont_common(pre_test_probabilities, weights_of_evidence):
    """Common-reference scores, not raw one-vs-rest LRs."""
    return common_reference_posterior(pre_test_probabilities, weights_of_evidence)


def stacked_chart_pre_post_cont_common(pre_test_probabilities, post_testing_probabilities, dx_names, weights_of_evidence, figsize=(6, 6)):
    """Plot validated probabilities; never normalize away an invalid result."""
    pre = validate_posterior(pre_test_probabilities, dx_names)
    post = validate_posterior(post_testing_probabilities, dx_names)
    scores = float_vector(weights_of_evidence, 'input log scores')
    if scores.shape != pre.shape:
        raise ValueError('input log scores and probabilities must match.')
    fitted = None
    if fitted is not None and float_vector(fitted, 'fitted OVR LRs').shape != pre.shape:
        raise ValueError('fitted OVR LRs and probabilities must match.')
    mode = "common-reference"
    fig, ax = plt.subplots(figsize=(7.3, max(4.5, 3.4 + .22 * len(pre))))
    bottom = np.zeros(2)
    for i, name in enumerate(dx_names):
        values = np.array([pre[i], post[i]])
        label = f'{name} | input {mode} LR {np.exp(scores[i]):.3g}'
        if fitted is not None:
            label += f' | fitted OVR LR {fitted[i]:.3g}'
        bars = ax.bar([0, 1], values, bottom=bottom, width=.32, label=label)
        for bar, value in zip(bars, values):
            if value >= .04:
                ax.text(bar.get_x()+bar.get_width()/2, bar.get_y()+value/2,
                        f'{value:.3f}', ha='center', va='center', color='white', fontsize=10)
        bottom += values
        ax.plot([0, 1], bottom, '--', color='gray', lw=.8)
    ax.set(xticks=[0, 1], xticklabels=['Before', 'After'], ylabel='Probability', ylim=(0, 1),
           title=('Coherent fit candidate (minimum unverified)' if mode == 'OVR' else 'Common-reference Bayes update'))
    ax.legend(loc='upper center', bbox_to_anchor=(.5, -.12), fontsize=8.5)
    fig.tight_layout()
    plt.show()


def generalized_predictor_wrapper_cont_common(diagnoses):
    """Relative prior weights and explicitly selected evidence semantics."""
    names = [d['name'] for d in diagnoses]
    raw = np.asarray([d['pretest'] for d in diagnoses], dtype=float)
    scores = float_vector([d['loglr'] for d in diagnoses], 'scores')
    mode = 'softmax_scores'
    if mode not in ('ovr_projection', 'softmax_scores'):
        raise ValueError('Unknown evidence mode.')
    priors = normalize_priors(raw, ovr=mode == 'ovr_projection')
    if len(scores) != len(priors):
        raise ValueError('scores and priors must have matching lengths.')
    fitted, fitted_log = None, None
    if mode == 'ovr_projection':
        result = require_candidate(ovr_project_posterior(priors, representable_exp(scores, 'input OVR LRs'), reg=0.))
        priors, post = result.diagnostics['priors'], result.posterior
        fitted, fitted_log = result.fitted_ovr_lr, result.diagnostics['fitted_log_ovr_lr']
        print('Input: OVR LRs at the displayed reference prior.')
    else:
        post = common_reference_posterior(priors, scores)
        print('Common-reference class log-likelihood scores. ')
    validate_posterior(post, names)
    print(f"{'Diagnosis':<20}{'Raw weight':<12}{'Prior':<12}{'Input log score':<18}{'Posterior':<12}")
    for i, name in enumerate(names):
        print(f"{name:<20}{raw[i]:<12.4g}{priors[i]:<12.4g}{scores[i]:<18.4g}{post[i]:<12.4g}")
    if fitted is not None:
        print('Input OVR LR | Fitted OVR LR | Fitted log OVR LR')
        for i, name in enumerate(names):
            print(f"{name}: {np.exp(scores[i]):.6g} | {fitted[i]:.6g} | {fitted_log[i]:.6g}")
    return dict(names=names, pre_test_probs=raw, norm_pre_test_probs=priors,
                weights_of_evidence=scores, predicted_probs=post,
                fitted_ovr_lr=fitted, fitted_loglr=fitted_log)


def create_diagnosis_widget_cont_common():
    diagnosis_count = widgets.IntSlider(
        value=1, min=1, max=10, step=1, description="Diagnoses"
    )
    container = VBox()
    chart_output = Output()

    def update_widgets(change):
        if change['name'] == 'value':
            container.children = []
            for i in range(change['new']):
                name_widget = widgets.Text(
                    value=f"Diagnosis {i+1}",
                    description="Name:"
                )
                
                # Pre-test probability slider from 0.0 to 1.0, 2-decimal readout
                pretest_widget = widgets.FloatSlider(
                    value=0.5,
                    min=0.0,
                    max=1.0,
                    step=0.01,
                    description="Prior weight:",
                    readout_format='.2f'
                )

                # Create the LogLR FloatLogSlider
                loglr_widget = widgets.FloatLogSlider(
                    value=1.0,            # Start with LR = 1
                    base=np.e,            # Base e
                    min=-5.0,             # Minimum LogLR
                    max=5.0,              # Maximum LogLR
                    step=0.1,             # Step size
                    description="LR vs ref:", # Label for the slider
                    readout=True,
                    readout_format=".2f"
                )

                diagnosis_box = VBox([
                    name_widget,
                    pretest_widget,
                    loglr_widget,
                ])
                container.children += (diagnosis_box,)

    diagnosis_count.observe(update_widgets, names='value')
    
    # Force the initial layout
    update_widgets({'name': 'value', 'new': diagnosis_count.value})

    def gather_inputs(_):
        diagnoses = []
        for diagnosis_box in container.children:
            name = diagnosis_box.children[0].value
            pretest = diagnosis_box.children[1].value
            loglr = np.log(diagnosis_box.children[2].value)  # Transform LR back to LogLR

            diagnoses.append({
                'name': name,
                'pretest': pretest,
                'loglr': loglr
            })

        chart_output.clear_output()
        with chart_output:
            try:
                results = generalized_predictor_wrapper_cont_common(diagnoses)
                stacked_chart_pre_post_cont_common(
                    results['norm_pre_test_probs'],
                    results['predicted_probs'],
                    results['names'],
                    results['weights_of_evidence']
                )
            except (ValueError, FloatingPointError, OverflowError) as exc:
                print(f"Cannot calculate: {exc}")

    calculate_button = widgets.Button(description="Calculate")
    calculate_button.on_click(gather_inputs)

    return HBox([VBox([diagnosis_count, container, calculate_button]), chart_output])


# Now create and display the revised widget
widget_cont_common = create_diagnosis_widget_cont_common()
display(widget_cont_common)
# Compatibility aliases; retained callbacks use the distinct names above.
naive_multinomial_predict = naive_multinomial_predict_cont_common
stacked_chart_pre_post = stacked_chart_pre_post_cont_common
generalized_predictor_wrapper = generalized_predictor_wrapper_cont_common
create_diagnosis_widget = create_diagnosis_widget_cont_common


### Two evidence objects

The preceding demonstration applies common-reference Bayes normalization to class log likelihood scores. It does not independently update binary slices and renormalize them. Inputs are relative prior weights and common-reference likelihood ratios.

The following demonstration additionally accepts OVR LRs, each defined against a prior-weighted complementary mixture for one observation. The prior, input labels and plot legend must match the selected mode. Raw OVR LRs cannot generally be fed into common-reference softmax.

Incompatible OVR inputs are fit with the retained weighted squared natural-log LR loss plus baseline-relative ridge. The result is a coherent candidate, not a demonstrated minimum or clinical calibration. At zero ridge, exact coherent inputs recover direct Bayes within tolerance. Positive ridge can shift them and depends on the chosen baseline.

For prior `(0.5,0.3,0.2)` and raw OVRs `(4,3,2)`, independent binary slices sum to `1.6958333`. Dividing by that sum lowers every slice, producing `(0.471744,0.331695,0.196560)` and different implied OVRs. This differs from normalizing `prior*raw_OVR`, which gives `(0.606061,0.272727,0.121212)`.

For repeated findings, combine class likelihood weights only under the stated conditional-independence model, or use history-conditioned likelihoods. Multiplying fixed-prior marginal OVRs is not a general joint-event rule; no sequential engine is implemented here.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from diagnostic_reasoning import (
    OVRProjectionResult, ovr_project_posterior, project_ovr_to_coherent,
    _project_ovr_to_coherent, common_reference_posterior, normalize_priors,
    validate_posterior, float_vector, representable_exp, require_candidate,
    CERTAINTY_ESTIMATES, qualitative_score,
)

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, widgets, VBox, HBox, Layout, Output, Label

# Optional styling
import scienceplots
plt.style.use('science')
plt.rcParams['text.usetex'] = False

# ============================================================
#  Bayes‑coherent OVR solver: concepts and data flow
#  -----------------------------------------------------------
#  • priors π_k (pre‑test probabilities) – sum to 1
#  • OVR LRs λ_k  (diagnosis k vs "not k") – possibly inconsistent
#  • We seek class log‑*scores* s_k (roughly, log likelihoods) so that:
#       model_logLR_k = s_k - log(∑_{j≠k} π_j e^{s_j}) + log(1-π_k)
#    matches the input log λ_k in least‑squares sense.
#  • Then posterior P_k ∝ π_k e^{s_k} is Bayes‑coherent by construction.
#  • L2 regularization on s controls numeric stability / overfitting.
# ============================================================








# ============================================================
#  Visualization: show pre vs post, annotate Input vs Fitted
# ============================================================
def stacked_chart_pre_post_cont_mixed(pre_test_probabilities, post_testing_probabilities, dx_names, input_loglr, fitted_ovr_lr=None, figsize=(6, 6)):
    """Plot validated probabilities; never normalize away an invalid result."""
    pre = validate_posterior(pre_test_probabilities, dx_names)
    post = validate_posterior(post_testing_probabilities, dx_names)
    scores = float_vector(input_loglr, 'input log scores')
    if scores.shape != pre.shape:
        raise ValueError('input log scores and probabilities must match.')
    fitted = fitted_ovr_lr
    if fitted is not None and float_vector(fitted, 'fitted OVR LRs').shape != pre.shape:
        raise ValueError('fitted OVR LRs and probabilities must match.')
    mode = ("OVR" if fitted is not None else "common-reference")
    fig, ax = plt.subplots(figsize=(7.3, max(4.5, 3.4 + .22 * len(pre))))
    bottom = np.zeros(2)
    for i, name in enumerate(dx_names):
        values = np.array([pre[i], post[i]])
        label = f'{name} | input {mode} LR {np.exp(scores[i]):.3g}'
        if fitted is not None:
            label += f' | fitted OVR LR {fitted[i]:.3g}'
        bars = ax.bar([0, 1], values, bottom=bottom, width=.32, label=label)
        for bar, value in zip(bars, values):
            if value >= .04:
                ax.text(bar.get_x()+bar.get_width()/2, bar.get_y()+value/2,
                        f'{value:.3f}', ha='center', va='center', color='white', fontsize=10)
        bottom += values
        ax.plot([0, 1], bottom, '--', color='gray', lw=.8)
    ax.set(xticks=[0, 1], xticklabels=['Before', 'After'], ylabel='Probability', ylim=(0, 1),
           title=('Coherent fit candidate (minimum unverified)' if mode == 'OVR' else 'Common-reference Bayes update'))
    ax.legend(loc='upper center', bbox_to_anchor=(.5, -.12), fontsize=8.5)
    fig.tight_layout()
    plt.show()


# ============================================================
#  Wrapper: read inputs, run either OVR projection (recommended)
#  or "common-reference softmax" (assumes shared denominator).
# ============================================================
def generalized_predictor_wrapper_cont_mixed(diagnoses, method="ovr_projection", reg=1e-6):
    """Relative prior weights and explicitly selected evidence semantics."""
    names = [d['name'] for d in diagnoses]
    raw = np.asarray([d['pretest'] for d in diagnoses], dtype=float)
    scores = float_vector([d['loglr'] for d in diagnoses], 'scores')
    mode = method
    if mode not in ('ovr_projection', 'softmax_scores'):
        raise ValueError('Unknown evidence mode.')
    priors = normalize_priors(raw, ovr=mode == 'ovr_projection')
    if len(scores) != len(priors):
        raise ValueError('scores and priors must have matching lengths.')
    fitted, fitted_log = None, None
    if mode == 'ovr_projection':
        result = require_candidate(ovr_project_posterior(priors, representable_exp(scores, 'input OVR LRs'), reg=reg))
        priors, post = result.diagnostics['priors'], result.posterior
        fitted, fitted_log = result.fitted_ovr_lr, result.diagnostics['fitted_log_ovr_lr']
        print('Input: OVR LRs at the displayed reference prior.')
    else:
        post = common_reference_posterior(priors, scores)
        print('Common-reference class log-likelihood scores. ')
    validate_posterior(post, names)
    print(f"{'Diagnosis':<20}{'Raw weight':<12}{'Prior':<12}{'Input log score':<18}{'Posterior':<12}")
    for i, name in enumerate(names):
        print(f"{name:<20}{raw[i]:<12.4g}{priors[i]:<12.4g}{scores[i]:<18.4g}{post[i]:<12.4g}")
    if fitted is not None:
        print('Input OVR LR | Fitted OVR LR | Fitted log OVR LR')
        for i, name in enumerate(names):
            print(f"{name}: {np.exp(scores[i]):.6g} | {fitted[i]:.6g} | {fitted_log[i]:.6g}")
    return dict(names=names, pre_test_probs=raw, norm_pre_test_probs=priors,
                weights_of_evidence=scores, predicted_probs=post,
                fitted_ovr_lr=fitted, fitted_loglr=fitted_log)
    

# ============================================================
#  Notebook widget: quick UI to experiment with priors and LRs
#  • "OVR coherent projection" (recommended) runs the solver
#  • "Common-reference softmax" assumes scores already share a
#    single denominator (no OVR mixture coupling enforced)
# ============================================================
def create_diagnosis_widget_cont_mixed():
    diagnosis_count = widgets.IntSlider(value=3, min=2, max=12, step=1, description="Diagnoses")
    method_dd = widgets.Dropdown(options=[('OVR coherent projection','ovr_projection'),
                                          ('Common-reference softmax','softmax_scores')],
                                 value='ovr_projection', description='Method')
    # Ridge strength slider (log-scale). ↑ reg = more shrinkage / stability.
    reg_slider = widgets.FloatLogSlider(value=1e-6, base=10, min=-12, max=-2, step=0.5,
                                        description="Reg (L2)", readout_format=".1e")
    zero_ridge = widgets.Checkbox(value=False, description="No ridge (reg=0)")

    container = VBox()
    chart_output = Output()

    def update_widgets(change):
        if change['name'] == 'value':
            children = []
            for i in range(change['new']):
                name_widget = widgets.Text(value=f"Diagnosis {i+1}", description="Name:")
                pretest_widget = widgets.FloatSlider(value=0.33, min=0.0, max=1.0, step=0.01,
                                                     description="Prior weight:", readout_format='.2f')
                # Slider shows LR on a log scale; we convert to log() at readout.
                lr_widget = widgets.FloatLogSlider(value=1.0, base=np.e, min=-5.0, max=5.0, step=0.1,
                                                   description=("OVR LR:" if method_dd.value == "ovr_projection" else "LR vs ref:"), readout=True, readout_format=".2f")
                children.append(VBox([name_widget, pretest_widget, lr_widget]))
            container.children = tuple(children)

    diagnosis_count.observe(update_widgets, names='value')
    update_widgets({'name': 'value', 'new': diagnosis_count.value})

    def update_mode(change=None):
        diagnosis_count.min = 2 if method_dd.value == 'ovr_projection' else 1
        for row in container.children:
            row.children[2].description = ('OVR LR:' if method_dd.value == 'ovr_projection' else 'LR vs ref:')
    method_dd.observe(update_mode, names='value')
    diagnosis_count.observe(update_mode, names='value')

    def gather_inputs(_):
        # Assemble inputs from the UI
        diagnoses = []
        for box in container.children:
            name = box.children[0].value
            pretest = box.children[1].value
            loglr = np.log(box.children[2].value)  # store log OVR LR
            diagnoses.append({'name': name, 'pretest': pretest, 'loglr': loglr})

        chart_output.clear_output()
        with chart_output:
            # Run chosen method and plot pre vs post; legend shows Input vs Fitted LRs
            try:
                results = generalized_predictor_wrapper_cont_mixed(diagnoses, method=method_dd.value, reg=(0. if zero_ridge.value else reg_slider.value))
                stacked_chart_pre_post_cont_mixed(results['norm_pre_test_probs'],
                                       results['predicted_probs'],
                                       results['names'],
                                       results['weights_of_evidence'],
                                       fitted_ovr_lr=results['fitted_ovr_lr'])
            except (ValueError, FloatingPointError, OverflowError) as exc:
                print(f"Cannot calculate: {exc}")

    calculate_button = widgets.Button(description="Calculate")
    calculate_button.on_click(gather_inputs)

    left = VBox([diagnosis_count, method_dd, VBox([reg_slider, zero_ridge]), container, calculate_button])
    return HBox([left, chart_output])


# Build & display widget
widget_cont_mixed = create_diagnosis_widget_cont_mixed()
display(widget_cont_mixed)
# Compatibility aliases; retained callbacks use the distinct names above.
stacked_chart_pre_post = stacked_chart_pre_post_cont_mixed
generalized_predictor_wrapper = generalized_predictor_wrapper_cont_mixed
create_diagnosis_widget = create_diagnosis_widget_cont_mixed


### One-vs-rest Coherent Projection

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from diagnostic_reasoning import (
    OVRProjectionResult, ovr_project_posterior, project_ovr_to_coherent,
    _project_ovr_to_coherent, common_reference_posterior, normalize_priors,
    validate_posterior, float_vector, representable_exp, require_candidate,
    CERTAINTY_ESTIMATES, qualitative_score,
)

# ==== OVR COHERENT PROJECTION (self-contained) =================================
# Assumes: inputs are one-vs-rest LRs (k vs not-k). Outputs: Bayes-coherent posterior.

import numpy as np, matplotlib.pyplot as plt
from ipywidgets import VBox, HBox, IntSlider, Text, FloatSlider, FloatLogSlider, Output, Button, Layout

# Optional, pretty style (safe if not installed)
try:
    import scienceplots; plt.style.use('science')
    plt.rcParams['text.usetex'] = False
except Exception:
    pass

# ---------- Core solver ----------





# ---------- Minimal plotting & UI ----------


def run_once_ovr_cont_ovr(names, priors, ovr_lrs, reg=1e-6):
    pri = normalize_priors(priors, ovr=True)
    if len(names) != len(pri):
        raise ValueError('names and priors must have the same length.')
    res = require_candidate(ovr_project_posterior(pri, ovr_lrs, reg=reg))
    pri = res.diagnostics['priors']
    print(f"{'Diagnosis':<16}{'Prior':<10}{'Input OVR LR':<15}{'Fitted OVR LR':<16}{'Fitted logLR':<14}{'Posterior':<10}")
    for i, name in enumerate(names):
        print(f"{name:<16}{pri[i]:<10.4g}{ovr_lrs[i]:<15.5g}{res.fitted_ovr_lr[i]:<16.5g}{res.diagnostics['fitted_log_ovr_lr'][i]:<14.5g}{res.posterior[i]:<10.4g}")
    plot_pre_post_cont_ovr(pri, res.posterior, names, np.log(ovr_lrs), res.fitted_ovr_lr)
    return res

def plot_pre_post_cont_ovr(pre, post, names, input_loglr, fitted_ovr_lr):
    """Plot validated probabilities; never normalize away an invalid result."""
    pre = validate_posterior(pre, names)
    post = validate_posterior(post, names)
    scores = float_vector(input_loglr, 'input log scores')
    if scores.shape != pre.shape:
        raise ValueError('input log scores and probabilities must match.')
    fitted = fitted_ovr_lr
    if fitted is not None and float_vector(fitted, 'fitted OVR LRs').shape != pre.shape:
        raise ValueError('fitted OVR LRs and probabilities must match.')
    mode = "OVR"
    fig, ax = plt.subplots(figsize=(7.3, max(4.5, 3.4 + .22 * len(pre))))
    bottom = np.zeros(2)
    for i, name in enumerate(names):
        values = np.array([pre[i], post[i]])
        label = f'{name} | input {mode} LR {np.exp(scores[i]):.3g}'
        if fitted is not None:
            label += f' | fitted OVR LR {fitted[i]:.3g}'
        bars = ax.bar([0, 1], values, bottom=bottom, width=.32, label=label)
        for bar, value in zip(bars, values):
            if value >= .04:
                ax.text(bar.get_x()+bar.get_width()/2, bar.get_y()+value/2,
                        f'{value:.3f}', ha='center', va='center', color='white', fontsize=10)
        bottom += values
        ax.plot([0, 1], bottom, '--', color='gray', lw=.8)
    ax.set(xticks=[0, 1], xticklabels=['Before', 'After'], ylabel='Probability', ylim=(0, 1),
           title=('Coherent fit candidate (minimum unverified)' if mode == 'OVR' else 'Common-reference Bayes update'))
    ax.legend(loc='upper center', bbox_to_anchor=(.5, -.12), fontsize=8.5)
    fig.tight_layout()
    plt.show()


def make_widget_ovr_cont_ovr():
    """
    Controls on the LEFT; output (table + plot) on the RIGHT.
    """
    # controls
    n_slider  = IntSlider(value=3, min=2, max=12, step=1, description="Diagnoses",
                          layout=Layout(width="280px"))
    reg_slider = FloatLogSlider(value=1e-6, base=10, min=-12, max=-2, step=0.5,
                                description="Reg (L2)", readout_format=".1e",
                                layout=Layout(width="280px"))
    zero_ridge = widgets.Checkbox(value=False, description="No ridge (reg=0)")

    # container for per-class rows (scrollable if long)
    box = VBox(layout=Layout(overflow="auto", max_height="420px", width="100%"))
    out = Output(layout=Layout(width="60%", border="1px solid #ddd", padding="8px",
                               min_height="520px"))
    # left panel (fixed width so right output sits beside it)
    left = VBox(layout=Layout(width="38%", min_width="340px"))

    def rebuild(_):
        rows = []
        for i in range(n_slider.value):
            name = Text(value=f"Diagnosis {i+1}", description="Name", layout=Layout(width="95%"))
            prior = FloatSlider(value=0.33, min=0, max=1, step=0.01, description="Prior weight",
                                readout_format=".2f", layout=Layout(width="95%"))
            lr = FloatLogSlider(value=1.0, base=np.e, min=-5, max=5, step=0.1,
                                description="OVR LR", readout_format=".2f", layout=Layout(width="95%"))
            rows.append(VBox([name, prior, lr], layout=Layout(padding="2px 0")))
        box.children = tuple(rows)

    n_slider.observe(rebuild, 'value')
    rebuild(None)

    # button action: compute and render to the RIGHT
    def compute(_):
        from math import log
        names, priors, lrs = [], [], []
        for c in box.children:
            names.append(c.children[0].value)
            priors.append(c.children[1].value)
            lrs.append(float(c.children[2].value))
        out.clear_output()
        with out:
            # assumes you already have run_once_ovr_cont_ovr(...) defined in this notebook
            try:
                run_once_ovr_cont_ovr(names, priors, lrs, reg=(0. if zero_ridge.value else float(reg_slider.value)))
            except (ValueError, FloatingPointError, OverflowError) as exc:
                print(f"Cannot calculate: {exc}")

    btn = Button(description="Calculate", layout=Layout(width="120px"))
    btn.on_click(compute)

    # assemble left and right
    left.children = (HBox([n_slider, VBox([reg_slider, zero_ridge])]), box, btn)
    ui = HBox([left, out], layout=Layout(align_items="flex-start", justify_content="space-between"))
    return ui

# re-render the widget
widget_cont_ovr = make_widget_ovr_cont_ovr()
display(widget_cont_ovr)
# ---------------------------------------------------------------------------------
# ===============================================================================
# Compatibility aliases; retained callbacks use the distinct names above.
run_once_ovr = run_once_ovr_cont_ovr
plot_pre_post = plot_pre_post_cont_ovr
make_widget_ovr = make_widget_ovr_cont_ovr


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from diagnostic_reasoning import (
    OVRProjectionResult, ovr_project_posterior, project_ovr_to_coherent,
    _project_ovr_to_coherent, common_reference_posterior, normalize_priors,
    validate_posterior, float_vector, representable_exp, require_candidate,
    CERTAINTY_ESTIMATES, qualitative_score,
)

# ==== COMMON-REFERENCE SOFTMAX (self-contained, cleaned layout) ================
# Assumes: inputs are scores s_k = log L_k - log C(x) with the SAME C(x) for all classes.
# Outputs: Bayes-coherent posterior via softmax(log π_k + s_k).  UI: controls (left), output (right)

import numpy as np, matplotlib.pyplot as plt
from ipywidgets import VBox, HBox, IntSlider, Text, FloatSlider, Output, Button, Layout

# Optional, pretty style (safe if not installed)
try:
    import scienceplots
    plt.style.use('science')
    plt.rcParams['text.usetex'] = False
except Exception:
    pass

# ---------- Core posterior ----------
def softmax_posterior_cont_scores(priors, scores):
    """Common-reference scores, not raw one-vs-rest LRs."""
    return common_reference_posterior(priors, scores)

# ---------- Plot: stacked bars + dashed connectors ----------
def plot_pre_post_cont_scores(pre, post, names, scores):
    """Plot validated probabilities; never normalize away an invalid result."""
    pre = validate_posterior(pre, names)
    post = validate_posterior(post, names)
    scores = float_vector(scores, 'input log scores')
    if scores.shape != pre.shape:
        raise ValueError('input log scores and probabilities must match.')
    fitted = None
    if fitted is not None and float_vector(fitted, 'fitted OVR LRs').shape != pre.shape:
        raise ValueError('fitted OVR LRs and probabilities must match.')
    mode = "common-reference"
    fig, ax = plt.subplots(figsize=(7.3, max(4.5, 3.4 + .22 * len(pre))))
    bottom = np.zeros(2)
    for i, name in enumerate(names):
        values = np.array([pre[i], post[i]])
        label = f'{name} | input {mode} LR {np.exp(scores[i]):.3g}'
        if fitted is not None:
            label += f' | fitted OVR LR {fitted[i]:.3g}'
        bars = ax.bar([0, 1], values, bottom=bottom, width=.32, label=label)
        for bar, value in zip(bars, values):
            if value >= .04:
                ax.text(bar.get_x()+bar.get_width()/2, bar.get_y()+value/2,
                        f'{value:.3f}', ha='center', va='center', color='white', fontsize=10)
        bottom += values
        ax.plot([0, 1], bottom, '--', color='gray', lw=.8)
    ax.set(xticks=[0, 1], xticklabels=['Before', 'After'], ylabel='Probability', ylim=(0, 1),
           title=('Coherent fit candidate (minimum unverified)' if mode == 'OVR' else 'Common-reference Bayes update'))
    ax.legend(loc='upper center', bbox_to_anchor=(.5, -.12), fontsize=8.5)
    fig.tight_layout()
    plt.show()

# ---------- Run once: print table + plot ----------
def run_once_softmax_cont_scores(names, priors, scores):
    pri = normalize_priors(priors)
    post = common_reference_posterior(pri, scores)
    validate_posterior(post, names)
    print('Common-reference scores; prior weights normalized to the displayed probabilities.')
    for name, p, score, q in zip(names, pri, scores, post):
        print(f'{name}: prior={p:.4g}, log score={score:.4g}, posterior={q:.4g}')
    plot_pre_post_cont_scores(pri, post, names, scores)
    return post

# ---------- Widget: controls on LEFT, output on RIGHT ----------
def make_widget_softmax_cont_scores():
    # Left controls
    n_slider = IntSlider(value=3, min=1, max=12, step=1, description="Diagnoses",
                         layout=Layout(width="280px"))
    box = VBox(layout=Layout(overflow="auto", max_height="420px", width="100%"))
    out = Output(layout=Layout(width="60%", border="1px solid #ddd", padding="8px",
                               min_height="520px"))
    left = VBox(layout=Layout(width="38%", min_width="340px"))

    def rebuild(_):
        rows = []
        for i in range(n_slider.value):
            name = Text(value=f"Diagnosis {i+1}", description="Name", layout=Layout(width="95%"))
            prior = FloatSlider(value=0.33, min=0, max=1, step=0.01, description="Prior weight",
                                readout_format=".2f", layout=Layout(width="95%"))
            # Score s_k: real-valued; interpret as log LR vs a shared baseline (or log-likelihood up to a constant)
            score = FloatSlider(value=0.0, min=-5, max=5, step=0.1, description="Score s_k",
                                readout_format=".2f", layout=Layout(width="95%"))
            rows.append(VBox([name, prior, score], layout=Layout(padding="2px 0")))
        box.children = tuple(rows)

    n_slider.observe(rebuild, 'value'); rebuild(None)

    def compute(_):
        names, priors, scores = [], [], []
        for c in box.children:
            names.append(c.children[0].value)
            priors.append(c.children[1].value)
            scores.append(c.children[2].value)
        out.clear_output()
        with out:
            try:
                run_once_softmax_cont_scores(names, priors, scores)
            except (ValueError, FloatingPointError, OverflowError) as exc:
                print(f"Cannot calculate: {exc}")

    btn = Button(description="Calculate", layout=Layout(width="120px"))
    btn.on_click(compute)

    # Assemble left/right panes
    left.children = (n_slider, box, btn)
    ui = HBox([left, out], layout=Layout(align_items="flex-start", justify_content="space-between"))
    return ui

# Launch widget
widget_cont_scores = make_widget_softmax_cont_scores()
display(widget_cont_scores)
# ===============================================================================
# Compatibility aliases; retained callbacks use the distinct names above.
softmax_posterior = softmax_posterior_cont_scores
plot_pre_post = plot_pre_post_cont_scores
run_once_softmax = run_once_softmax_cont_scores
make_widget_softmax = make_widget_softmax_cont_scores
